# Data Wrangling with Polars

This notebook covers data wrangling techniques using **Polars**, a fast DataFrame library built on Apache Arrow.

We work with two datasets from the `data/` folder:
- `ingatlan-listings.parquet` — 9M+ price observations per property per day  
- `ingatlan-details.parquet` — property-level attributes

In [ ]:
import polars as pl

In [ ]:
listings_df = pl.read_parquet("data/ingatlan-listings.parquet")
listings_df

In [ ]:
details_df = pl.read_parquet("data/ingatlan-details.parquet")
details_df

In [ ]:
listings_ldf = pl.scan_parquet("data/ingatlan-listings.parquet")
listings_ldf  # shows the query plan, not the data

In [ ]:
details_ldf = pl.scan_parquet("data/ingatlan-details.parquet")
details_ldf

In [ ]:
print(listings_ldf.explain(optimized=True))

In [ ]:
listings_ldf.collect()

## Q1 — Filter: properties in Budapest

In [ ]:
details_df.filter(pl.col("city") == "Budapest")

## Q2 — Filter: expensive listings

In [ ]:
listings_df.filter(pl.col("price") > 500_000)

## Q3 — Sort: 10 largest properties

In [ ]:
details_df.sort("m2", descending=True).head(10)

## Q4 — Select: keep a subset of columns

In [ ]:
details_df.select("id", "city", "m2", "price")

## Q5 — Filter + sort: cheap and small properties

In [ ]:
details_df.filter((pl.col("m2") < 40) & (pl.col("price") < 100_000)).sort("price")

## Q6 — Group by: number of properties per city

In [ ]:
city_counts = details_df.group_by("city").len().sort("len", descending=True)

In [ ]:
import plotly.express as px

px.bar(
    city_counts.head(20),
    x="len",
    y="city",
    orientation="h",
    title="Top 20 Cities by Property Count",
    labels={"len": "Number of Properties", "city": "City"},
)

## Q7 — Group by + agg: average price and area per city

In [ ]:
(
    details_df.group_by("city")
    .agg(
        avg_price=pl.col("price").mean(),
        avg_m2=pl.col("m2").mean(),
    )
    .sort("avg_price", descending=True)
)

## Q8 — String filter: Budapest district properties

In [ ]:
details_df.filter(
    pl.col("loc").str.contains("kerület"),
)

## Q9 — Derived column: has balcony flag

In [ ]:
details_df.with_columns(
    has_balcony=pl.col("balcony").is_not_null()
    & (pl.col("balcony").str.len_chars() > 0)
)

## Q10 — Time filter: listings recorded in 2026

In [ ]:
listings_df.filter(pl.col("day").dt.year() == 2026)

## Q11 — Join: attach city to every listing

In [ ]:
listings_with_city = listings_df.join(
    details_df.select("id", "city"),
    on="id",
    how="left",
)
listings_with_city

## Q12 — Time series: average listing price per calendar month

In [ ]:
monthly_avg = (
    listings_df.with_columns(
        month=pl.col("day").dt.truncate("1mo"),
    )
    .group_by("month")
    .agg(
        avg_price=pl.col("price").mean(),
    )
    .sort("month")
)
monthly_avg

In [ ]:
px.line(
    monthly_avg,
    x="month",
    y="avg_price",
    title="Average Listing Price per Month",
    labels={"month": "Month", "avg_price": "Avg Price (000 HUF)"},
)

## Q13 — Derived column: price per square metre

In [ ]:
(
    details_df.with_columns(
        price_per_m2=pl.col("price") / pl.col("m2"),
    )
    .sort("price_per_m2", descending=True)
    .head(10)
)

## Q14 — String extract: numeric balcony area

In [ ]:
details_df.with_columns(
    balcony_m2=pl.col("balcony").str.extract(r"(\d+\.?\d*)").cast(pl.Float64),
)

## Q15 — Window function: price rank within each city

In [ ]:
(
    details_df.with_columns(
        city_price_rank=pl.col("price")
        .rank(method="dense", descending=True)
        .over("city"),
    ).sort(["city", "city_price_rank"])
)

## Q16 — Time series: most recent price per property

In [ ]:
listings_df.sort("day", descending=True).group_by("id").first()

## Q17 — Join + string: top 10 cities by average price-per-m²

In [ ]:
(
    details_df.with_columns(
        price_per_m2=pl.col("price") / pl.col("m2"),
    )
    .group_by("city")
    .agg(
        avg_price_per_m2=pl.col("price_per_m2").mean(),
    )
    .sort("avg_price_per_m2", descending=True)
    .head(10)
)

## Q18 — String parse + join + time series: monthly average price by room count

In [ ]:
room_counts = details_df.with_columns(
    room_count=pl.col("rooms").str.extract(r"^(\d+)").cast(pl.Int64),
).select("id", "room_count")

monthly_room = (
    listings_df.join(room_counts, on="id", how="left")
    .filter(pl.col("room_count").is_between(1, 5))
    .with_columns(
        month=pl.col("day").dt.truncate("1mo"),
    )
    .group_by("month", "room_count")
    .agg(
        avg_price=pl.col("price").mean(),
    )
    .sort("month", "room_count")
)

In [ ]:
px.line(
    monthly_room,
    x="month",
    y="avg_price",
    color="room_count",
    title="Monthly Average Listing Price by Room Count",
    labels={
        "month": "Month",
        "avg_price": "Avg Price (000 HUF)",
        "room_count": "Rooms",
    },
)

## Q19 — Window function: properties with the most price drops

In [ ]:
(
    listings_df.sort(["id", "day"])
    .with_columns(
        prev_price=pl.col("price").shift(1).over("id"),
    )
    .with_columns(
        is_drop=(pl.col("price") < pl.col("prev_price")),
    )
    .group_by("id")
    .agg(
        drop_count=pl.col("is_drop").sum(),
    )
    .sort("drop_count", descending=True)
    .head(10)
)

## Q20 — Multi-step: cities with the biggest average price drop 2025 → 2026

In [ ]:
city_lookup = details_df.select("id", "city")

avg_by_city_year = (
    listings_df.join(city_lookup, on="id", how="left")
    .with_columns(year=pl.col("day").dt.year())
    .filter(pl.col("year").is_in([2025, 2026]))
    .group_by("city", "year")
    .agg(avg_price=pl.col("price").mean())
)

avg_2025 = avg_by_city_year.filter(pl.col("year") == 2025).select(
    "city", avg_price_2025=pl.col("avg_price")
)
avg_2026 = avg_by_city_year.filter(pl.col("year") == 2026).select(
    "city", avg_price_2026=pl.col("avg_price")
)

(
    avg_2025.join(avg_2026, on="city", how="inner")
    .with_columns(
        price_drop=pl.col("avg_price_2025") - pl.col("avg_price_2026"),
    )
    .sort("price_drop", descending=True)
    .head(10)
)

## Q21 — Window + rolling: first day a property's rolling price crossed below the global median

In [ ]:
global_median = listings_df["price"].median()

(
    listings_df.sort(["id", "day"])
    .with_columns(
        rolling_avg=pl.col("price").rolling_mean(window_size=7).over("id"),
    )
    .filter(
        pl.col("rolling_avg").is_not_null() & (pl.col("rolling_avg") < global_median)
    )
    .group_by("id")
    .agg(
        first_cross_day=pl.col("day").min(),
    )
    .sort("first_cross_day")
)

## Q22 — Z-score anomaly detection per property

In [ ]:
(
    listings_df.with_columns(
        mean_price=pl.col("price").mean().over("id"),
        std_price=pl.col("price").std().over("id"),
    )
    .filter(pl.col("std_price") > 0)
    .with_columns(
        z_score=(pl.col("price") - pl.col("mean_price")) / pl.col("std_price"),
    )
    .sort(pl.col("z_score").abs(), descending=True)
    .head(20)
    .select("id", "day", "price", "z_score")
)

## Q23 — String regex: Budapest district median price per m²

In [ ]:
districts = (
    details_df.filter(pl.col("city") == "Budapest")
    .with_columns(
        district=pl.col("loc").str.extract(r"^([IVX]+)\. kerület"),
        price_per_m2=pl.col("price") / pl.col("m2"),
    )
    .filter(pl.col("district").is_not_null())
    .group_by("district")
    .agg(
        median_price_per_m2=pl.col("price_per_m2").median(),
        listing_count=pl.len(),
    )
    .sort("median_price_per_m2", descending=True)
)

In [ ]:
px.bar(
    districts,
    x="district",
    y="median_price_per_m2",
    title="Budapest Districts — Median Price per m²",
    labels={"district": "District", "median_price_per_m2": "Median Price/m² (000 HUF)"},
)

## Q24 — Cohort analysis: average price by months since first listing

In [ ]:
first_listing_month = listings_df.group_by("id").agg(
    cohort_month=pl.col("day").min().dt.truncate("1mo"),
)
(
    listings_df.join(first_listing_month, on="id")
    .with_columns(
        month=pl.col("day").dt.truncate("1mo"),
    )
    .with_columns(
        months_since_first=(
            (pl.col("month").dt.year() - pl.col("cohort_month").dt.year()) * 12
            + pl.col("month").dt.month()
            - pl.col("cohort_month").dt.month()
        ).cast(pl.Int32),
    )
    .filter(pl.col("months_since_first").is_between(0, 5))
    .group_by("cohort_month", "months_since_first")
    .agg(
        avg_price=pl.col("price").mean(),
        property_count=pl.col("id").n_unique(),
    )
    .sort("cohort_month", "months_since_first")
)

## Q25 — Price spike detection with neighbour comparison

In [ ]:
(
    listings_df.sort(["id", "day"])
    .with_columns(
        prev_price=pl.col("price").shift(1).over("id"),
        next_price=pl.col("price").shift(-1).over("id"),
    )
    .with_columns(
        is_spike=(pl.col("price") > pl.col("prev_price"))
        & (pl.col("price") > pl.col("next_price")),
    )
    .group_by("id")
    .agg(
        spike_count=pl.col("is_spike").sum(),
    )
    .sort("spike_count", descending=True)
    .head(10)
    .join(
        details_df.select("id", "city", "loc"),
        on="id",
        how="left",
    )
)

## Q26 — Pairwise monthly price correlation between cities

In [ ]:
top10_cities = (
    details_df.group_by("city")
    .len()
    .sort("len", descending=True)
    .head(10)
    .get_column("city")
    .to_list()
)

monthly_city = (
    listings_df.join(details_df.select("id", "city"), on="id", how="left")
    .filter(pl.col("city").is_in(top10_cities))
    .with_columns(month=pl.col("day").dt.truncate("1mo"))
    .group_by("city", "month")
    .agg(avg_price=pl.col("price").mean())
)

(
    monthly_city.join(
        monthly_city.rename({"city": "city_b", "avg_price": "avg_price_b"}),
        on="month",
    )
    .filter(pl.col("city") < pl.col("city_b"))
    .group_by("city", "city_b")
    .agg(
        correlation=pl.corr("avg_price", "avg_price_b"),
    )
    .sort("correlation", descending=True)
)

## Q27 — First day a property crossed its city median from below to above (2025)

In [ ]:
city_median = details_df.group_by("city").agg(
    city_median_price=pl.col("price").median(),
)
(
    listings_df.join(details_df.select("id", "city"), on="id", how="left")
    .join(city_median, on="city", how="left")
    .sort(["id", "day"])
    .with_columns(
        above_median=(pl.col("price") > pl.col("city_median_price")),
    )
    .with_columns(
        prev_above_median=pl.col("above_median").shift(1).over("id"),
    )
    .filter(
        pl.col("above_median")
        & (pl.col("prev_above_median") == False)
        & (pl.col("day").dt.year() == 2025)
    )
    .group_by("id")
    .agg(
        first_cross_up_day=pl.col("day").min(),
    )
    .sort("first_cross_up_day")
)

## Q28 — Price acceleration: top 10 most erratic properties

In [ ]:
(
    listings_df.sort(["id", "day"])
    .with_columns(
        velocity=(pl.col("price") - pl.col("price").shift(1).over("id")).cast(
            pl.Float64
        ),
    )
    .with_columns(
        acceleration=pl.col("velocity") - pl.col("velocity").shift(1).over("id"),
    )
    .group_by("id")
    .agg(
        mean_abs_acceleration=pl.col("acceleration").abs().mean(),
        obs_count=pl.len(),
    )
    .filter(pl.col("obs_count") >= 60)
    .sort("mean_abs_acceleration", descending=True)
    .head(10)
    .join(
        details_df.select("id", "city", "loc"),
        on="id",
        how="left",
    )
)

## Q29 — Cohort retention rate

In [ ]:
(
    listings_df.group_by("id")
    .agg(
        first_day=pl.col("day").min(),
        last_day=pl.col("day").max(),
    )
    .with_columns(
        cohort_month=pl.col("first_day").dt.truncate("1mo"),
        survived_90d=((pl.col("last_day") - pl.col("first_day")).dt.total_days() >= 90),
    )
    .group_by("cohort_month")
    .agg(
        total=pl.len(),
        survived=pl.col("survived_90d").sum(),
    )
    .with_columns(
        retention_rate=pl.col("survived").cast(pl.Float64) / pl.col("total"),
    )
    .sort("cohort_month")
)

## Q30 — Listing gap fraction: missing days on the market

In [ ]:
(
    listings_df.group_by("id")
    .agg(
        first_day=pl.col("day").min(),
        last_day=pl.col("day").max(),
        observed_days=pl.len(),
    )
    .with_columns(
        expected_days=(
            (pl.col("last_day") - pl.col("first_day")).dt.total_days() + 1
        ).cast(pl.Int64),
    )
    .with_columns(
        gap_fraction=(
            (pl.col("expected_days") - pl.col("observed_days")).cast(pl.Float64)
            / pl.col("expected_days")
        ),
    )
    .filter(pl.col("observed_days") >= 30)
    .sort("gap_fraction", descending=True)
    .head(10)
    .join(
        details_df.select("id", "city", "loc"),
        on="id",
        how="left",
    )
)